# NoLA · RESISC45 matched experiment
**Original CDE vs. template-aware complete linkage at cosine similarity 0.75.**

Upload this notebook to Colab → **Runtime → Change runtime type → GPU** → review Settings → **Run all**. No EuroSAT bundle or local repository is needed. Defaults run only frozen CLIP validation. A validation-only run produces reports but no final test score.

**Split provenance:** [NoLA's pinned release](https://github.com/fazliimam/NoLA/tree/caec2722bba84bd4807256645c5cb75d96d7ed12) has an official RESISC45 adapter, but no public `split_resisc45.json` was found in the release. If you have the author's JSON, set `NOLA_SPLIT_JSON` (and optionally `NOLA_SPLIT_SHA256`). Otherwise this notebook executes the pinned adapter's DTD 99/0/1 split on sorted JPEG names, using seed 7777. It never uses the adapter's `val = test` shortcut: when val is absent or aliases test it holds out 20% per class from adapter train with seed 7778, retaining the adapter test. The default split is **24,930 / 6,255 / 315**; it is deterministic and **not** a claim to reproduce the authors' exact partition. The 315-item test is small (7 per class). Set `ALLOW_ADAPTER_GENERATED = False` to require the supplied file. The pinned JPEG mirror supplies images only; its own split files are not used. Cite [NWPU-RESISC45](https://doi.org/10.1109/JPROC.2017.2675998) and [NoLA](https://bmvc2025.bmva.org/proceedings/281/) in reports.


In [ ]:
# Edit settings before running. Rerun from the top after any change.
from pathlib import Path
WORK = Path('/content/nola_resisc45')
DRIVE = False                   # True: mount Drive for reports and restartable checkpoints
REPORT_ROOT = Path('/content/nola_resisc45/reports') # e.g. /content/drive/MyDrive/NoLA-RESISC45
IMAGE_ZIP = None                 # None: verified pinned mirror; or path to an equivalent NWPU JPEG ZIP
IMAGE_DIR_OVERRIDE = None        # Alternative: existing canonical NWPU-RESISC45/ directory
NOLA_SPLIT_JSON = None           # Optional path to your author-supplied split_resisc45.json
NOLA_SPLIT_SHA256 = None         # Optional known hash for the supplied file
ALLOW_ADAPTER_GENERATED = True   # False: require an author-supplied JSON; never silently fall back
RUN_ADAPTATION = False           # DINO teacher + visual prompt/LayerNorm training (both arms)
FINAL_TEST = False               # Explicit release; final cell only, after selected stage(s) finish
SEED, SPLIT_SEED = 7777, 7777
EXTRACT_BATCH, TEXT_BATCH, WORKERS = 64, 128, 2
MAX_EPOCHS, TEACHER_EPOCHS, PATIENCE = 50, 50, 15
MICRO, EFFECTIVE = 32, 512        # Keep effective batch fixed for matched comparison
EPOCHS_THIS_SESSION = None       # Optional cap on new epoch pairs per invocation, not on selection
BOOTSTRAP_REPS = 2000


## 1 · Runtime and verified inputs
This checks for CUDA and downloads the pinned JPEG archive, sources and model weights as needed. `DRIVE = True` mounts Drive; also set `REPORT_ROOT` under `/content/drive/MyDrive/` to persist checkpoints and reports. Keep `WORK` on the Colab VM for image I/O. Do not use two writers for the same run directory.

In [ ]:
import os, sys, subprocess
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # before CUDA setup
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab, restart the session, and rerun from the top.')
if DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
if str(REPORT_ROOT).startswith('/content/drive/') and not DRIVE:
    raise ValueError('Set DRIVE=True for a Drive report location.')
subprocess.run([sys.executable,'-m','pip','-q','install','ftfy==6.3.1','regex==2024.11.6'],check=True)
print('CUDA:', torch.cuda.get_device_name(), '| PyTorch:', torch.__version__)


In [ ]:
import ast, hashlib, json, os, random, re, shutil, subprocess, sys, zipfile
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import PurePosixPath

NOLA_REV = 'caec2722bba84bd4807256645c5cb75d96d7ed12'
CLIP_REV = 'd05afc436d78f1c48dc0dbf8e5980a9d471f35f6'
DINO_REV = '7c446df5b9f45747937fb0d72314eb9f7b66930a'
NOLA_URL = f'https://raw.githubusercontent.com/fazliimam/NoLA/{NOLA_REV}/'
CLIP_URL = f'https://raw.githubusercontent.com/openai/CLIP/{CLIP_REV}/clip/'
DINO_URL = f'https://raw.githubusercontent.com/facebookresearch/dino/{DINO_REV}/'
IMAGE_URL = 'https://huggingface.co/datasets/isaaccorley/resisc45/resolve/883edc0eee77b2c84225472f10f126e3ed83fa6e/NWPU-RESISC45.zip'
IMAGE_SHA = 'beeecd0b63656290ae6d65cf7763185b0c1c4c54a753ef8088d6fba3faaf1f53'
CLIP_SHA = '40d365715913c9da98579312b702a82c18be219cc2a73407c4526f58eba950af'
DINO_SHA = 'bf34ad0f424b9029b593e8dc3ed553bf26e88bcba0d32bf3e62a6209cb64c85e'
SOURCES = {
    'resisc45.py': (NOLA_URL+'datasets/resisc45.py','f38556ac9456daff25a9d1e0617d1b73367a0c9c327cb81185fdd2818f07766c'),
    'dtd.py': (NOLA_URL+'datasets/dtd.py','97360089123688c6c6635be6399f89a36e83caab2e82e2d15d01ae8b8c199de8'),
    'oxford_pets.py': (NOLA_URL+'datasets/oxford_pets.py','223f04a757ab92d534a5329758818ad265bc7de9febd3707ca0506b094519ed0'),
    'data_utils.py': (NOLA_URL+'utils/data_utils.py','c7baf0b1b486ec644318abdc3a98f59b321f7165d0dfab5dc88d9980dfe0e364'),
    'RESISC45.json': (NOLA_URL+'descriptions/generic/RESISC45.json','5884ecb4ddecf52b758dcf2479472496a3ee3c5edbe3fcd20c8b868058fb42c3'),
    'model.py': (CLIP_URL+'model.py','9902cbe5ee90a1da2aa3e6f043e8a23dc1f8831193b963785c9af03d5c7bef2c'),
    'simple_tokenizer.py': (CLIP_URL+'simple_tokenizer.py','d1b09f10ed3d1e2c343619cb98cbfacb92363dd8607526a43aa7fe2d60d15bad'),
    'bpe_simple_vocab_16e6.txt.gz': (CLIP_URL+'bpe_simple_vocab_16e6.txt.gz','924691ac288e54409236115652ad4aa250f48203de50a9e4722a6ecd48d6804a'),
    'vision_transformer.py': (DINO_URL+'vision_transformer.py','b1f998d5f49ab43666b9fc6d007c5f6540c3ead2e8645e144f74045f63ff44d7'),
    'dino_utils.py': (DINO_URL+'utils.py','962a97e1acda1c986dfd921275325e4083f9016dddadcf018f1ccf03fa600eab'),
    'ViT-B-32.pt': (f'https://openaipublic.azureedge.net/clip/models/{CLIP_SHA}/ViT-B-32.pt',CLIP_SHA),
    'dino_vitbase16_pretrain.pth': ('https://dl.fbaipublicfiles.com/dino/dino_vitbase16_pretrain/dino_vitbase16_pretrain.pth',DINO_SHA),
}
WRAPPERS = ('Satellite photo: {}','A satellite photo: {}','The satellite photo: {}',
            'A centered satellite photo: {}','This is an aerial image: {}','An aerial image: {}')
THRESHOLD, TAU = .75, .05
ARMS = ('original_cde', 'template_aware_075')

def digest(p):
    with Path(p).open('rb') as f: return hashlib.file_digest(f, 'sha256').hexdigest()

def fetch(url, path, sha):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.is_file() and digest(path) == sha: return path
    tmp = path.with_name(path.name+'.part')
    tmp.unlink(missing_ok=True)
    subprocess.run(['curl','-fLsS','--retry','2','--connect-timeout','20','--max-time','1200',url,'-o',str(tmp)],check=True)
    if digest(tmp) != sha:
        tmp.unlink()
        raise ValueError(f'SHA-256 mismatch: {path}')
    tmp.replace(path)
    return path

def source(name): return fetch(*((SOURCES[name][0], WORK/'sources'/name, SOURCES[name][1])))

def write_json(path, obj):
    tmp = path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')
    tmp.replace(path)

def read_json(path): return json.loads(Path(path).read_text(encoding='utf-8'))

def get_literal(tree, name):
    return ast.literal_eval(next(n.value for n in tree.body if isinstance(n, ast.Assign)
                                 and any(isinstance(t, ast.Name) and t.id == name for t in n.targets)))

@dataclass
class Datum:
    impath: str
    label: int
    classname: str

def upstream_method(tree, class_name, method, env):
    cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == class_name)
    fn = next(n for n in cls.body if isinstance(n, ast.FunctionDef) and n.name == method)
    fn.decorator_list = []
    exec(compile(ast.Module(body=[fn],type_ignores=[]), f'{class_name}.{method}', 'exec'), env)
    return env[method]

def image_manifest(root, keys):
    expected = {f'{k}/{k}_{i:03d}.jpg' for k in keys for i in range(1,701)}
    found = {p.relative_to(root).as_posix() for p in root.glob('*/*.jpg') if p.is_file()}
    if expected != found:
        raise ValueError(f'Image inventory mismatch: {len(expected-found)} missing, {len(found-expected)} extra')
    return expected

def prepare_data():
    if NOLA_SPLIT_JSON is None and not ALLOW_ADAPTER_GENERATED:
        raise ValueError('Provide NOLA_SPLIT_JSON or explicitly enable ALLOW_ADAPTER_GENERATED.')
    ds = WORK/'data'/'resisc45'
    ds.mkdir(parents=True, exist_ok=True)
    trees = {n:ast.parse(source(n).read_text()) for n in ('resisc45.py','dtd.py','oxford_pets.py','data_utils.py')}
    names = get_literal(trees['resisc45.py'], 'NEW_CNAMES')
    templates = get_literal(trees['data_utils.py'], 'ds_specific_templates')['RESISC45']
    descriptions = read_json(source('RESISC45.json'))
    assert set(descriptions) == set(names) and len(names) == 45 and len(templates) == 6
    assert all(len(v) == 50 and all(isinstance(s,str) and s.strip() for s in v) for v in descriptions.values())
    archive_sha = None
    if IMAGE_DIR_OVERRIDE is None:
        archive = Path(IMAGE_ZIP) if IMAGE_ZIP else fetch(IMAGE_URL, WORK/'NWPU-RESISC45.zip', IMAGE_SHA)
        archive_sha = digest(archive) if IMAGE_ZIP else IMAGE_SHA
        root = ds/'NWPU-RESISC45'
        marker = ds/'image_archive_sha256.txt'
        if marker.exists() and marker.read_text() != archive_sha:
            shutil.rmtree(root, ignore_errors=True)
            marker.unlink()
        with zipfile.ZipFile(archive) as z:
            members = {}
            for item in z.infolist():
                rel = PurePosixPath(item.filename)
                if rel.suffix != '.jpg' or rel.parent.name not in names or rel.is_absolute() or '..' in rel.parts:
                    continue
                key = '/'.join(rel.parts[-2:])
                if key in members: raise ValueError(f'Duplicate image: {key}')
                members[key] = item
            expected = {f'{k}/{k}_{i:03d}.jpg' for k in names for i in range(1,701)}
            if set(members) != expected: raise ValueError('ZIP does not contain the canonical 31,500 JPEG inventory.')
            for key, item in members.items():
                dest = root/key
                if marker.exists() and dest.is_file() and dest.stat().st_size == item.file_size: continue
                dest.parent.mkdir(parents=True, exist_ok=True)
                dest.write_bytes(z.read(item))
            marker.write_text(archive_sha)
    else:
        root = Path(IMAGE_DIR_OVERRIDE)
    manifest = image_manifest(root, names)
    # Run the exact release's splitter/read_split with small Dassl shims.
    # Sorted directory enumeration makes DTD's RNG independent of extraction order.
    def list_clean(path):
        p = Path(path)
        return sorted(names if p == root else (x.name for x in p.glob('*.jpg')))
    rng = random.Random(SPLIT_SEED)
    env = dict(os=os, Datum=Datum, listdir_nohidden=list_clean, random=rng, read_json=read_json)
    load = upstream_method(trees['oxford_pets.py'],'OxfordPets','read_split',env)
    if NOLA_SPLIT_JSON is not None:
        raw = Path(NOLA_SPLIT_JSON)
        raw_sha = digest(raw)
        if NOLA_SPLIT_SHA256 and raw_sha != NOLA_SPLIT_SHA256: raise ValueError('Supplied split hash mismatch')
        input_json = read_json(raw)
        if not all(isinstance(p,str) and p in manifest for part in ('train','val','test')
                   for p,_,_ in input_json.get(part,[])):
            raise ValueError('Noncanonical or invalid path in supplied split.')
        train, val, test = load(raw, str(root))
        origin = 'user_supplied_NoLA_JSON'
    else:
        split = upstream_method(trees['dtd.py'],'DescribableTextures','read_and_split_data',env)
        train, val, test = split(str(root), new_cnames=names)
        raw_sha = None
        origin = 'NoLA_adapter_generated_not_author_manifest'
    def rows(seq): return [[Path(x.impath).relative_to(root).as_posix(), int(x.label), x.classname] for x in (seq or [])]
    base = dict(train=rows(train),val=rows(val),test=rows(test))
    # An author manifest may alias validation to test. Discard the alias, never select on it.
    val_paths, test_paths = ({r[0] for r in base[k]} for k in ('val','test'))
    if val_paths and val_paths == test_paths and len(base['val']) == len(base['test']):
        base['val'] = []
    all_rows = sum(base.values(), [])
    paths = [r[0] for r in all_rows]
    if len(paths) != len(set(paths)) or set(paths) != manifest:
        raise ValueError('Split must cover the inventory once; train/val/test overlap or missing images detected.')
    key_by_id = {label: p.split('/')[0] for p,label,_ in all_rows}
    if set(key_by_id) != set(range(45)) or len(set(key_by_id.values())) != 45:
        raise ValueError('Invalid class IDs in split')
    if not all(key_by_id[y] == p.split('/')[0] and c in (key_by_id[y],names[key_by_id[y]]) for p,y,c in all_rows):
        raise ValueError('Class path/name/label disagreement')
    base_snapshot = {k:list(v) for k,v in base.items()}
    carved = not base['val']
    if carved:
        # Stratified, predeclared 20% of adapter train; adapter test membership unchanged.
        strata = defaultdict(list)
        for i, (_,y,_) in enumerate(base['train']): strata[y].append(i)
        vrng = random.Random(SPLIT_SEED+1)
        held = {i for y in sorted(strata) for i in vrng.sample(strata[y],round(.2*len(strata[y])))}
        old = base['train']
        base['train'] = [r for i,r in enumerate(old) if i not in held]
        base['val'] = [r for i,r in enumerate(old) if i in held]
    if not all({x[1] for x in v} == set(range(45)) for v in base.values()):
        raise ValueError('Each effective split must contain all 45 classes.')
    if origin.startswith('NoLA_adapter_generated'):
        assert {k:len(v) for k,v in base.items()} == dict(train=24930,val=6255,test=315)
    return root, base, base_snapshot, [key_by_id[i] for i in range(45)], names, templates, descriptions, dict(
        split_origin=origin, supplied_split_sha256=raw_sha, validation_carved=carved,
        image_archive_sha256=archive_sha, supplied_image_dir=str(root) if IMAGE_DIR_OVERRIDE else None)


## 2 · Fixed text banks and paired validation
- **Original CDE:** unit-normalize embeddings of all 50 exact released descriptions plus the six released NoLA class-name templates, mean, then unit-normalize per class.
- **Template-aware:** six fixed view wrappers (same strings as the EuroSAT run) for each of the 50 exact descriptions; unit-normalize each text embedding, mean its six views, then unit-normalize the description vector. Cluster independently per class by complete linkage with cosine distance ≤0.25. Use raw cluster means, **not** normalized cluster centroids, and weight each by its description multiplicity in log-mean-exp (τ=0.05). This is the compound EuroSAT comparison, not a clustering-only ablation: the text inputs and scoring both change.
- Both arms share the same frozen ViT-B/32 features and images. Report top-1/macro accuracy, per-class counts, paired fixes/losses, a class-stratified paired bootstrap interval and exact two-sided McNemar p-value. Intervals quantify image-level variation for a fixed run, not variability over seeds. Adapted validation statistics are descriptive because the checkpoints are selected on that split.


In [ ]:
import contextlib, importlib.metadata, importlib.util, math, time
import numpy as np
import scipy
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import pdist, squareform
from scipy.stats import binomtest
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms as T
from PIL import Image, ImageFilter

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN = (.48145466,.4578275,.40821073)
STD = (.26862954,.26130258,.27577711)
WEAK = T.Compose([T.Resize(224,interpolation=T.InterpolationMode.BICUBIC),T.CenterCrop(224),T.ToTensor(),T.Normalize(MEAN,STD)])

class Blur:
    def __call__(self, im): return im.filter(ImageFilter.GaussianBlur(random.uniform(.1,2)))

STRONG = T.Compose([T.RandomResizedCrop(224,scale=(.2,1.)),T.RandomHorizontalFlip(),
    T.RandomApply([T.ColorJitter(.4,.4,.4,.1)],p=.8),
    T.RandomPerspective(distortion_scale=.2,p=.5,interpolation=T.InterpolationMode.BICUBIC),
    T.ColorJitter(.4,.4,.4,.1),Blur(),T.ToTensor(),T.Normalize(MEAN,STD)])

class Images(Dataset):
    def __init__(self,root,rows,tx): self.root,self.rows,self.tx = root,rows,tx
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        with Image.open(self.root/self.rows[i][0]) as im: return self.tx(im.convert('RGB')), i

def loader(rows, tx=WEAK, batch=None, shuffle_seed=None):
    return DataLoader(Images(IMAGE_ROOT,rows,tx), batch_size=batch or EXTRACT_BATCH,
        shuffle=shuffle_seed is not None, num_workers=WORKERS, pin_memory=DEVICE.type=='cuda',
        generator=torch.Generator().manual_seed(SEED if shuffle_seed is None else shuffle_seed))

def module(name,path):
    spec = importlib.util.spec_from_file_location(name,path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def load_clip():
    cm = module('pinned_clip_model',source('model.py'))
    with torch.no_grad():
        jit = torch.jit.load(str(source('ViT-B-32.pt')),map_location='cpu').eval()
        model = cm.build_model(jit.state_dict()).float().eval().requires_grad_(False).to(DEVICE)
    source('bpe_simple_vocab_16e6.txt.gz')
    tok = module('pinned_clip_tokenizer',source('simple_tokenizer.py')).SimpleTokenizer()
    return model,tok

def tokenize(strings,tok):
    ids = torch.zeros(len(strings),77,dtype=torch.long)
    start, end = (tok.encoder[t] for t in ('<|startoftext|>','<|endoftext|>'))
    for i,s in enumerate(strings):
        seq = tok.encode(s)
        if len(seq)>75: raise ValueError(f'CLIP context exceeded without truncation: {s!r}')
        seq = [start,*seq,end]
        ids[i,:len(seq)] = torch.tensor(seq)
    return ids

@torch.inference_mode()
def text_vectors(model,tok,prompts):
    return torch.cat([F.normalize(model.encode_text(x.to(DEVICE)).float(),dim=-1).cpu()
                      for x in tokenize(prompts,tok).split(TEXT_BATCH)])

def cluster(v):
    # v rows already unit-normalized. Complete linkage on cosine distance, class by class.
    d = pdist(v.double().numpy(),metric='cosine')
    labels = fcluster(linkage(d,method='complete'),1-THRESHOLD,criterion='distance')
    groups = sorted([np.flatnonzero(labels == k).tolist() for k in np.unique(labels)])
    assert sorted(i for g in groups for i in g) == list(range(len(v)))
    mat = squareform(d)
    assert all(mat[np.ix_(g,g)].max() <= 1-THRESHOLD+1e-10 for g in groups)
    return groups

def make_bank(model,tok,keys,names,templates,desc):
    baseline, proto, sizes, audit = [], [], [], []
    for key in keys:
        descriptions = desc[key]
        plain = [t.format(key.replace('_',' ')) for t in templates]
        wrapped = [w.format(d) for d in descriptions for w in WRAPPERS]
        original = text_vectors(model,tok,descriptions+plain)
        v = F.normalize(text_vectors(model,tok,wrapped).reshape(len(descriptions),len(WRAPPERS),-1).mean(1),dim=-1)
        groups = cluster(v)
        means = torch.stack([v[g].mean(0) for g in groups]) # Do NOT normalize cluster means.
        counts = [len(g) for g in groups]
        torch.testing.assert_close((means*torch.tensor(counts)[:,None]).sum(0)/len(v),v.mean(0))
        baseline.append(F.normalize(original.mean(0),dim=0))
        proto.append(means)
        sizes.append(counts)
        audit.append(dict(key=key,classname=names[key],descriptions=descriptions,original_class_templates=plain,
                          groups=groups,multiplicities=counts,raw_mean_norms=means.norm(dim=1).tolist()))
    return dict(broad=torch.stack(baseline),prototypes=torch.cat(proto),sizes=sizes,audit=audit)

class BankScorer:
    def __init__(self,bank):
        self.bank = bank
        self.broad = bank['broad'].to(DEVICE)
        self.prototypes = bank['prototypes'].to(DEVICE)
        groups = bank['sizes']; c,w = len(groups),max(map(len,groups))
        self.index = torch.zeros(c,w,dtype=torch.long,device=DEVICE)
        self.log_size = torch.full((c,w),-float('inf'),device=DEVICE)
        self.log_total = torch.tensor([sum(g) for g in groups],device=DEVICE).float().log()
        j = 0
        for y,g in enumerate(groups):
            self.index[y,:len(g)] = torch.arange(j,j+len(g),device=DEVICE)
            self.log_size[y,:len(g)] = torch.tensor(g,device=DEVICE).float().log()
            j += len(g)
    def weights(self,arm): return self.broad if arm == ARMS[0] else self.prototypes
    def __call__(self,x,arm,w=None,raw=False):
        if w is None: w=self.weights(arm)
        if arm == ARMS[0]: return x@w.T
        norm = x.norm(dim=1,keepdim=True).clamp_min(1e-8) if raw else 1.
        logits = ((x/norm)@w.T)/TAU
        return norm*TAU*(torch.logsumexp(logits[:,self.index]+self.log_size,dim=-1)-self.log_total)

@torch.inference_mode()
def frozen_features(model,rows):
    out = torch.cat([F.normalize(model.encode_image(x.to(DEVICE,non_blocking=True)).float(),dim=1).cpu()
                     for x,_ in loader(rows)])
    assert out.shape==(len(rows),512) and out.isfinite().all()
    return out

@torch.inference_mode()
def frozen_scores(features):
    return {a:torch.cat([SCORER(x.to(DEVICE),a).cpu() for x in features.split(1024)]) for a in ARMS}

def atomic_save(obj,path):
    tmp = path.with_name(path.name+'.tmp')
    torch.save(obj,tmp)
    tmp.replace(path)

def load_tensor(path): return torch.load(path,map_location='cpu',weights_only=True)

def cached(path,compute):
    if path.is_file(): return load_tensor(path)
    obj = compute()
    atomic_save(obj,path)
    return obj


In [ ]:
import csv, platform

def csv_rows(path,rows):
    if not rows: return
    tmp = path.with_name(path.name+'.tmp')
    with tmp.open('w',newline='',encoding='utf-8') as f:
        w=csv.DictWriter(f,fieldnames=list(rows[0]))
        w.writeheader(); w.writerows(rows)
    tmp.replace(path)

def truth(rows): return torch.tensor([r[1] for r in rows],dtype=torch.long)

def paired_report(stage,split,rows,pred):
    # Rows are the common, unshuffled manifest; predictions are paired per image.
    y=truth(rows); a,b=(pred[k].cpu() for k in ARMS); c=len(KEYS)
    assert a.shape==b.shape==y.shape and all(((p>=0)&(p<c)).all() for p in (a,b))
    ca,cb = a==y,b==y
    fixed,lost = ~ca&cb,ca&~cb
    prefix = OUT/f'{stage}_{split}'
    def row(mask):
        n=int(mask.sum()); na,nb,nf,nl=(int((v&mask).sum()) for v in (ca,cb,fixed,lost))
        return dict(n=n,original_correct=na,template_correct=nb,original_pct=100*na/n,
                    template_pct=100*nb/n,delta_pp=100*(nb-na)/n,fixed=nf,lost=nl)
    per = [dict(class_id=i,class_key=k,classname=NAMES[k],**row(y==i)) for i,k in enumerate(KEYS)]
    overall = row(torch.ones_like(y,dtype=torch.bool))
    rng=np.random.default_rng(SEED)
    d=(cb.to(torch.int8)-ca.to(torch.int8)).numpy()
    # Paired bootstrap stratified by true class (fixed observed class composition).
    boot=np.zeros(BOOTSTRAP_REPS,dtype=float)
    for i in range(c):
        z=d[y.numpy()==i]; n=len(z)
        p=np.bincount(z+1,minlength=3)/n
        sample=rng.multinomial(n,p,size=BOOTSTRAP_REPS)
        boot += sample[:,2]-sample[:,0]
    overall.update(stage=stage,split=split,seed=SEED,bootstrap_reps=BOOTSTRAP_REPS,
        delta_pp_ci95=(100*np.quantile(boot,[.025,.975])/len(y)).tolist(),
        exact_mcnemar_p=float(binomtest(overall['fixed'],overall['fixed']+overall['lost'],.5).pvalue)
           if overall['fixed']+overall['lost'] else 1.0,
        original_macro_pct=sum(r['original_pct'] for r in per)/c,
        template_macro_pct=sum(r['template_pct'] for r in per)/c)
    write_json(prefix.with_name(prefix.name+'_summary.json'),overall)
    csv_rows(prefix.with_name(prefix.name+'_per_class.csv'),per)
    for arm, p in zip(ARMS,(a,b)):
        matrix=torch.bincount(y*c+p,minlength=c*c).reshape(c,c).tolist()
        csv_rows(prefix.with_name(prefix.name+f'_{arm}_confusion.csv'),
                 [dict(true_class=k,**dict(zip(KEYS,v))) for k,v in zip(KEYS,matrix)])
    csv_rows(prefix.with_name(prefix.name+'_predictions.csv'),
        [dict(path=r[0],truth_id=r[1],original_pred=int(aa),template_pred=int(bb),
              fixed=bool(f),lost=bool(l)) for r,aa,bb,f,l in zip(rows,a,b,fixed,lost)])
    print(json.dumps(overall,indent=2)); return overall

def setup():
    global IMAGE_ROOT,SPLITS,KEYS,NAMES,TEMPLATES,DESC,OUT,CACHE,CONFIG,CONFIG_ID
    assert BOOTSTRAP_REPS>0 and WORKERS>0 and 0<MICRO<=EFFECTIVE and EFFECTIVE%MICRO==0
    assert min(MAX_EPOCHS,TEACHER_EPOCHS,PATIENCE,EXTRACT_BATCH,TEXT_BATCH)>0
    assert EPOCHS_THIS_SESSION is None or EPOCHS_THIS_SESSION>0
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.backends.cuda.matmul.allow_tf32=False
    torch.backends.cudnn.allow_tf32=False
    torch.backends.cudnn.benchmark=False
    torch.backends.cudnn.deterministic=True
    torch.use_deterministic_algorithms(True)
    IMAGE_ROOT,SPLITS,base,KEYS,NAMES,TEMPLATES,DESC,meta=prepare_data()
    def json_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
    if IMAGE_DIR_OVERRIDE and meta['image_archive_sha256'] is None:
        # Content fingerprint only. Do not decode or score test images during setup.
        h=hashlib.sha256()
        for rel,_,_ in sorted(sum(SPLITS.values(),[])):
            h.update(rel.encode()); h.update(bytes.fromhex(digest(IMAGE_ROOT/rel)))
        meta['image_tree_sha256']=h.hexdigest()
    CONFIG=dict(protocol='resisc45-matched-v1', noLA_commit=NOLA_REV, clip_commit=CLIP_REV,dino_commit=DINO_REV,
        sources={n:dict(url=u,sha256=h) for n,(u,h) in SOURCES.items()},
        eurosat_runner_sha256='a325812eff52c4a2e66c12ed0a4361da307898c100be91b0d1fa225d222fed2b',
        archive_url=IMAGE_URL if IMAGE_ZIP is None and IMAGE_DIR_OVERRIDE is None else None,
        **meta,split_seed=SPLIT_SEED,seed=SEED,split_algorithm='upstream DTD defaults with sorted JPG enumeration; stratified 20% of train if validation absent/aliases test',
        base_split_sha256=json_hash(base), split_sha256=json_hash(SPLITS),
        split_sizes={k:len(v) for k,v in SPLITS.items()}, class_keys=KEYS, class_name_map=NAMES,
        threshold=THRESHOLD,tau=TAU,wrappers=WRAPPERS,class_templates=TEMPLATES,
        clip_preprocess='bicubic resize short side 224, center crop 224, CLIP RGB mean/std',
        dino_input='same weak CLIP transform as EuroSAT runner',
        max_epochs=MAX_EPOCHS,teacher_epochs=TEACHER_EPOCHS,patience=PATIENCE,
        effective_batch=EFFECTIVE,microbatch=MICRO,extract_batch=EXTRACT_BATCH,text_batch=TEXT_BATCH,workers=WORKERS,
        anchors_per_class=math.floor(.2*len(SPLITS['train'])/len(KEYS)), anchor_policy='first nonempty top-1/2/3 rank; descending confidence, stable ties, class order dedup',
        teacher_batch=32,teacher_lr=.001,student_lr=.004,student_tokens=16,label_smoothing=.1,
        precision='eager float32 frozen CLIP; AMP fp16 GPU adaptation', bootstrap_reps=BOOTSTRAP_REPS,
        runtime=dict(python=platform.python_version(),torch=str(torch.__version__),torchvision=str(torchvision.__version__),
        numpy=np.__version__,scipy=scipy.__version__,pillow=importlib.metadata.version('Pillow'),
        ftfy=importlib.metadata.version('ftfy'),regex=importlib.metadata.version('regex'),
        cuda=torch.version.cuda,gpu=torch.cuda.get_device_name() if DEVICE.type=='cuda' else None))
    CONFIG_ID=json_hash(CONFIG)[:16]
    OUT=REPORT_ROOT/CONFIG_ID
    CACHE=WORK/'cache'/CONFIG_ID
    for p in (OUT,CACHE): p.mkdir(parents=True,exist_ok=True)
    write_json(OUT/'config.json',CONFIG)
    write_json(OUT/'split_train_val.json',{k:SPLITS[k] for k in ('train','val')})
    print('Run',CONFIG_ID,'|',meta['split_origin'],'|',CONFIG['split_sizes'],'| reports:',OUT)

def frozen_validation():
    global CLIP,SCORER
    CLIP,tok=load_clip()
    bank=cached(CACHE/'text_bank.pt',lambda:make_bank(CLIP,tok,KEYS,NAMES,TEMPLATES,DESC))
    SCORER=BankScorer(bank)
    write_json(OUT/'prompt_groups.json',dict(class_order=KEYS,wrappers=WRAPPERS,
        threshold=THRESHOLD,tau=TAU,classes=bank['audit']))
    atomic_save({k:bank[k] for k in ('broad','prototypes','sizes')},OUT/'text_banks.pt')
    csv_rows(OUT/'prototypes.csv',[dict(class_id=i,class_key=k,released_descriptions=len(DESC[k]),
        original_prompts=len(DESC[k])+len(TEMPLATES),wrapped_prompts=len(DESC[k])*len(WRAPPERS),
        original_prototypes=1,template_prototypes=len(g),sizes=';'.join(map(str,g)))
        for i,(k,g) in enumerate(zip(KEYS,bank['sizes']))])
    rows=SPLITS['val']
    features=cached(CACHE/'clip_val.pt',lambda:frozen_features(CLIP,rows))
    stats=paired_report('frozen_clip','val',rows,{a:x.argmax(1) for a,x in frozen_scores(features).items()})
    write_json(OUT/'frozen_validation_lock.json',dict(config_id=CONFIG_ID,arms=ARMS,selection='fixed settings; no sweep',stats=stats))


In [ ]:
setup()
frozen_validation()


## 3 · Optional matched adaptation (off by default)
The EuroSAT runner's schedule is ported to 45 classes: first-nonempty top-3 class anchor buckets, DINO ViT-B/16 and 768→256→45 Adam teacher, then 16 visual prompt tokens, visual LayerNorm and the arm's classifier with AdamW, FP16 AMP, identical per-epoch shuffle and strong transforms, and effective batch 512. The two arms have the same schedule and inputs; the classifier parametrizations and pseudo-targets differ by design. DINO uses the same weak **CLIP** pixel normalization as the validated EuroSAT runner. Training labels are used only in anchor audit outputs, not for training targets. Selection is highest validation correct per arm (earliest tie); 15 non-improvements or 50 epochs by default.

Set `RUN_ADAPTATION = True` and choose persistent `REPORT_ROOT` before you start. This may take multiple GPU sessions; use `EPOCHS_THIS_SESSION` to pause after a number of epoch rounds without changing the selection rule. Resume by rerunning from the top with the same settings, GPU type and dependency versions. Each teacher and each student's latest+best epoch state are saved atomically; a disconnect during an epoch can cost that epoch. Reducing `MICRO` (a divisor of 512) is an OOM option **before** starting a matched run; a changed configuration gets a new directory.

In [ ]:
def amp():
    return torch.autocast('cuda',dtype=torch.float16) if DEVICE.type=='cuda' else contextlib.nullcontext()

def load_dino():
    old = sys.modules.get('utils')
    sys.modules['utils'] = module('nola_pinned_dino_utils',source('dino_utils.py'))
    try: mod = module('nola_pinned_dino_vit',source('vision_transformer.py'))
    finally:
        if old is None: sys.modules.pop('utils',None)
        else: sys.modules['utils']=old
    model=mod.vit_base(patch_size=16,num_classes=0)
    model.load_state_dict(load_tensor(source('dino_vitbase16_pretrain.pth')),strict=True)
    return model.eval().requires_grad_(False).to(DEVICE)

@torch.inference_mode()
def dino_features(model,rows):
    out=[]
    for x,_ in loader(rows):
        with amp(): v=model(x.to(DEVICE,non_blocking=True))
        out.append(v.float().cpu())
    v=torch.cat(out)
    assert v.shape==(len(rows),768) and v.isfinite().all()
    return v

def select_anchors(scores,k,nclasses):
    order=torch.argsort(scores,dim=1,descending=True,stable=True)[:,:3]
    conf=scores.gather(1,order)
    ids,rank={},[]
    for c in range(nclasses):
        for r in range(3):
            ix=torch.where(order[:,r]==c)[0]
            if len(ix):
                top=ix[torch.argsort(conf[ix,r],descending=True,stable=True)[:k]]
                for j in top.tolist(): ids.setdefault(j,c)
                rank.append(dict(rank=r+1,candidates=len(ix),before_dedup=len(top)))
                break
        else: raise RuntimeError(f'No top-3 anchor candidate for class {c}')
    return torch.tensor(list(ids)),torch.tensor(list(ids.values())),rank

def fit_teacher(x,v,clip_scores):
    scale=CLIP.logit_scale.detach().exp().cpu()
    ix,y,ranks=select_anchors((scale*clip_scores).softmax(1),CONFIG['anchors_per_class'],len(KEYS))
    torch.manual_seed(SEED)
    rng=torch.Generator().manual_seed(SEED)
    head=nn.Sequential(nn.Linear(768,256),nn.ReLU(True),nn.Linear(256,len(KEYS))).to(DEVICE)
    opt=torch.optim.Adam(head.parameters(),lr=.001)
    x,v=x.to(DEVICE),v.to(DEVICE)
    for _ in range(TEACHER_EPOCHS):
        for idx in torch.randperm(len(ix),generator=rng).split(32):
            opt.zero_grad(set_to_none=True)
            loss=F.cross_entropy(head(x[ix[idx]]),y[idx].to(DEVICE))
            loss.backward(); opt.step()
    with torch.no_grad():
        return dict(targets=head(x).argmax(1).cpu(),val_pred=head(v).argmax(1).cpu(),ids=ix,labels=y,
                    ranks=ranks,updates=TEACHER_EPOCHS*math.ceil(len(ix)/32),
                    head={k:p.detach().cpu() for k,p in head.state_dict().items()})

def teachers():
    train,val=SPLITS['train'],SPLITS['val']
    paths={a:OUT/f'teacher-{a}.pt' for a in ARMS}
    if any(not p.is_file() for p in paths.values()):
        c=cached(CACHE/'clip_train.pt',lambda:frozen_features(CLIP,train))
        clip_scores=frozen_scores(c)
        need=not all((CACHE/f'dino_{k}.pt').is_file() for k in ('train','val'))
        dino=load_dino() if need else None
        x=cached(CACHE/'dino_train.pt',lambda:dino_features(dino,train))
        v=cached(CACHE/'dino_val.pt',lambda:dino_features(dino,val))
        del dino
        for a,p in paths.items():
            if not p.is_file(): atomic_save(fit_teacher(x,v,clip_scores[a]),p)
    result={a:load_tensor(p) for a,p in paths.items()}
    anchor_rows=[]; t=truth(train)
    for a,b in result.items():
        ix,y=b['ids'],b['labels']; correct=t[ix]==y
        csv_rows(OUT/f'{a}_anchors.csv',[dict(path=train[j][0],pseudo_id=int(l),audit_truth_id=int(t[j])) for j,l in zip(ix.tolist(),y)])
        for i,k in enumerate(KEYS):
            mask=y==i
            anchor_rows.append(dict(arm=a,class_id=i,class_key=k,**b['ranks'][i],anchor_count=int(mask.sum()),
                                    audit_correct=int((correct&mask).sum())))
    csv_rows(OUT/'anchor_summary.csv',anchor_rows)
    write_json(OUT/'teacher_updates.json',{a:b['updates'] for a,b in result.items()})
    paired_report('dino_teacher','val',val,{a:b['val_pred'] for a,b in result.items()})
    return result

def prompted(x,p=None):
    v=CLIP.visual; x=v.conv1(x).flatten(2).transpose(1,2)
    x=torch.cat((v.class_embedding.expand(len(x),1,-1).to(x.dtype),x),1)
    x=v.ln_pre(x+v.positional_embedding.to(x.dtype))
    if p is not None:x=torch.cat((x[:,:1],p.expand(len(x),-1,-1).to(x.dtype),x[:,1:]),1)
    x=v.transformer(x.transpose(0,1)).transpose(0,1)
    return v.ln_post(x[:,0])@v.proj

def init_student(arm):
    torch.manual_seed(SEED)
    with torch.no_grad():
        for n,p in PARAMS.items(): p.copy_(INITIAL[n])
    width=CLIP.visual.class_embedding.numel(); bound=math.sqrt(6/(3*16*16+width))
    p=nn.Parameter(torch.empty(1,16,width).uniform_(-bound,bound).to(DEVICE))
    w=nn.Parameter(SCORER.weights(arm).clone())
    opt=torch.optim.AdamW([dict(params=[p,w,*[v for n,v in PARAMS.items() if 'bias' not in n]],weight_decay=.01),
        dict(params=[v for n,v in PARAMS.items() if 'bias' in n],weight_decay=0.)],lr=.004,betas=(.9,.999))
    return p,w,opt,torch.amp.GradScaler('cuda',init_scale=512.,enabled=DEVICE.type=='cuda')

def restore(s,p,w,opt=None,scaler=None):
    with torch.no_grad():
        p.copy_(s['prompt']); w.copy_(s['weights'])
        for n,v in PARAMS.items():v.copy_(s['norm'][n])
    if opt is not None:opt.load_state_dict(s['optimizer']); scaler.load_state_dict(s['scaler'])

@torch.inference_mode()
def student_predict(rows,arm,p,w):
    output=[]
    for x,_ in loader(rows):
        with amp(): f=prompted(x.to(DEVICE,non_blocking=True),p)
        output.append(SCORER(f.float(),arm,w,raw=True).argmax(1).cpu())
    return torch.cat(output)

def train_epoch(arm,targets,old):
    p,w,opt,scaler=init_student(arm)
    if old is not None: restore(old['latest'],p,w,opt,scaler)
    e=1 if old is None else old['epoch']+1
    train,val=SPLITS['train'],SPLITS['val']
    start=time.monotonic(); n=updates=skips=0; losses=[]
    opt.zero_grad(set_to_none=True)
    for x,ix in loader(train,STRONG,MICRO,SEED+e):
        group_size=min(EFFECTIVE,len(train)-(n//EFFECTIVE)*EFFECTIVE)
        with amp(): f=prompted(x.to(DEVICE,non_blocking=True),p)
        loss=F.cross_entropy(SCORER(f.float(),arm,w,raw=True),targets[ix].to(DEVICE),
                            label_smoothing=.1,reduction='sum')/group_size
        scaler.scale(loss).backward(); n+=len(ix); losses.append(loss.detach())
        if n%EFFECTIVE==0 or n==len(train):
            scale=scaler.get_scale(); scaler.step(opt); scaler.update(); skips+=int(scaler.get_scale()<scale)
            opt.zero_grad(set_to_none=True); updates+=1
    assert n==len(train) and updates==math.ceil(n/EFFECTIVE) and skips<updates
    assert torch.isfinite(torch.stack(losses)).all() and all(x.isfinite().all() for x in [p,w,*PARAMS.values()])
    pred=student_predict(val,arm,p,w)
    correct=int((pred==truth(val)).sum())
    snapshot=dict(epoch=e,prompt=p.detach().cpu().clone(),weights=w.detach().cpu().clone(),
                  norm={k:v.detach().cpu().clone() for k,v in PARAMS.items()},pred=pred,correct=correct)
    latest=dict(**snapshot,optimizer=opt.state_dict(),scaler=scaler.state_dict())
    improved=old is None or correct>old['best']['correct']
    stale=0 if improved else old['stale']+1
    log=dict(arm=arm,epoch=e,n_train=n,optimizer_attempts=updates,amp_skips=skips,
             optimizer_updates=updates-skips,val_correct=correct,val_n=len(val),stale=stale,
             seconds=round(time.monotonic()-start,1))
    print(json.dumps(log),flush=True)
    return dict(config_id=CONFIG_ID,arm=arm,epoch=e,latest=latest,
        best=snapshot if improved else old['best'],stale=stale,history=(old['history'] if old else [])+[log])

def finished(state): return state is not None and (state['epoch']>=MAX_EPOCHS or state['stale']>=PATIENCE)

def configure_student():
    global PARAMS,INITIAL,STUDENT_MODEL
    if globals().get('STUDENT_MODEL') is CLIP:
        with torch.no_grad():
            for n,p in PARAMS.items(): p.copy_(INITIAL[n])
        return
    PARAMS={n:p for n,p in CLIP.named_parameters() if n.startswith('visual.') and 'ln' in n}
    assert PARAMS
    for p in PARAMS.values():p.requires_grad_(True)
    INITIAL={n:p.detach().cpu().clone() for n,p in PARAMS.items()}
    STUDENT_MODEL=CLIP

def adaptation_validation():
    assert read_json(OUT/'frozen_validation_lock.json')['config_id']==CONFIG_ID
    release=OUT/'test_release.json'
    if release.exists() and 'adapted_clip' not in read_json(release)['stages']:
        raise RuntimeError('This run already released frozen-only test results; do not extend it post hoc.')
    configure_student()
    t=teachers()
    with torch.no_grad():
        z=next(iter(loader(SPLITS['train'][:2])))[0].to(DEVICE)
        torch.testing.assert_close(prompted(z),CLIP.encode_image(z),rtol=1e-6,atol=1e-6)
    paths={a:OUT/f'adapt-{a}.pt' for a in ARMS}
    states={a:load_tensor(p) if p.is_file() else None for a,p in paths.items()}
    assert all(s is None or (s['config_id']==CONFIG_ID and s['arm']==a) for a,s in states.items())
    rounds=0
    while not all(finished(s) for s in states.values()):
        if EPOCHS_THIS_SESSION is not None and rounds>=EPOCHS_THIS_SESSION: break
        next_e=min((s['epoch'] if s else 0)+1 for s in states.values() if not finished(s))
        for a in ARMS:
            s=states[a]
            if not finished(s) and (s['epoch'] if s else 0)+1==next_e:
                states[a]=train_epoch(a,t[a]['targets'],s)
                atomic_save(states[a],paths[a])
        rounds+=1
    csv_rows(OUT/'student_epoch_log.csv',sorted([r for s in states.values() if s for r in s['history']],
             key=lambda r:(r['epoch'],r['arm'])))
    configure_student()
    if not all(finished(s) for s in states.values()):
        print('Paused at epoch boundary. Resume with unchanged settings (session cap can change). Test remains sealed.')
        return
    selection={a:dict(epoch=s['best']['epoch'],val_correct=s['best']['correct'],checkpoint_sha256=digest(paths[a]),
                      latest_epoch=s['epoch'],stale=s['stale']) for a,s in states.items()}
    paired_report('adapted_clip','val',SPLITS['val'],{a:s['best']['pred'] for a,s in states.items()})
    write_json(OUT/'adaptation_validation_lock.json',dict(config_id=CONFIG_ID,selection=selection,
               rule='highest validation correct per arm; earliest tie; both arms stopped'))


In [ ]:
if RUN_ADAPTATION:
    adaptation_validation()
else:
    print('Adaptation disabled; frozen validation complete.')


## 4 · Explicit test release and report export
Keep `FINAL_TEST = False` until you have committed to the complete stage plan. If you want adapted test results, enable adaptation and complete **both** arms' validation selection before release. With `FINAL_TEST = True`, the notebook evaluates the fixed frozen banks and, where selected, the locked best adapted checkpoints. The successful release cannot be extended to a different stage plan in the same run. Test images are not decoded or embedded before this cell (the archive is acquired and its inventory validated during setup).

The report ZIP contains config/source hashes, train/val manifests, prompt/group and prototype audits, paired summaries, per-image predictions, and class/confusion CSVs. Adaptation adds anchor audit and optimizer update logs. Checkpoints are stored separately in the run directory and are **not** in the report ZIP. Download the ZIP from the Colab Files pane (path printed below).

In [ ]:
def final_test():
    assert read_json(OUT/'frozen_validation_lock.json')['config_id']==CONFIG_ID
    completed_lock=OUT/'adaptation_validation_lock.json'
    want_adapt = RUN_ADAPTATION or completed_lock.is_file()
    if not want_adapt and any((OUT/f'adapt-{a}.pt').is_file() for a in ARMS):
        raise RuntimeError('Adaptation checkpoints exist but selection is not complete. Finish adaptation before releasing test.')
    states={}
    if want_adapt:
        if not completed_lock.is_file(): raise RuntimeError('Both arms must complete validation selection before test.')
        lock=read_json(completed_lock)
        assert lock['config_id']==CONFIG_ID
        for a in ARMS:
            p=OUT/f'adapt-{a}.pt'
            if digest(p)!=lock['selection'][a]['checkpoint_sha256']: raise ValueError(f'Selected checkpoint changed: {a}')
            state=load_tensor(p)
            assert finished(state) and state['best']['epoch']==lock['selection'][a]['epoch']
            states[a]=state
    plan=dict(config_id=CONFIG_ID,stages=['frozen_clip']+(['adapted_clip'] if want_adapt else []),
        frozen_lock_sha256=digest(OUT/'frozen_validation_lock.json'),
        adaptation_lock_sha256=digest(completed_lock) if want_adapt else None)
    release=OUT/'test_release.json'
    if release.exists() and read_json(release)!=plan:
        raise RuntimeError('Test release already recorded for a different stage plan; do not reuse test for post-hoc selection.')
    write_json(release,plan)
    if (OUT/'test_complete.json').exists():
        print('Identical test release already complete; read the existing artifacts.')
        return
    rows=SPLITS['test']
    if want_adapt:
        # Verify best validation predictions again using the selected snapshots before decoding test images.
        configure_student()
        for a,s in states.items():
            p,w,_,_=init_student(a); restore(s['best'],p,w)
            torch.testing.assert_close(student_predict(SPLITS['val'],a,p,w),s['best']['pred'])
        configure_student()
    features=frozen_features(CLIP,rows)
    paired_report('frozen_clip','test',rows,{a:x.argmax(1) for a,x in frozen_scores(features).items()})
    if want_adapt:
        pred={}
        for a,s in states.items():
            p,w,_,_=init_student(a); restore(s['best'],p,w)
            pred[a]=student_predict(rows,a,p,w)
        paired_report('adapted_clip','test',rows,pred)
        configure_student()
    write_json(OUT/'test_complete.json',plan)

def export_reports():
    dest=WORK/f'NoLA-RESISC45-{CONFIG_ID}-report.zip'
    # Reports only. Checkpoints live in OUT, optionally on Drive; not part of this export.
    with zipfile.ZipFile(dest,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUT.iterdir()):
            if p.suffix in {'.csv','.json'} or p.name=='text_banks.pt': z.write(p,p.name)
    print('Report ZIP:',dest,'| Checkpoints:',OUT)
    return dest


In [ ]:
REPORT_ZIP = export_reports()
if FINAL_TEST:
    final_test()
    REPORT_ZIP = export_reports()
else:
    print('Test remains sealed (FINAL_TEST = False).')
